# HW14 — Mini-RAG с FAISS

## 2.3.1 Импорты и seed

In [1]:
import numpy as np
import pandas as pd
import random
import faiss
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

os.makedirs("artifacts", exist_ok=True)

## 2.3.2 База знаний и первичный анализ

База знаний посвящена базовым понятиям машинного обучения, NLP и retrieval-систем.

Она включает короткие определения ключевых терминов (FAISS, embeddings, recall, precision и др.), 
что делает её удобной для демонстрации retrieval и mini-RAG.

Размер базы небольшой (~20 документов).

In [2]:

documents = [
    "Python is widely used in data science, machine learning, and automation tasks.",
    "Pandas is a Python library used for data manipulation and analysis.",
    "NumPy provides support for arrays and numerical computations.",
    "FAISS is a library developed by Facebook for efficient similarity search.",
    "Machine learning involves training models on data to make predictions.",
    "Deep learning is a subset of machine learning based on neural networks.",
    "Embeddings convert text into dense vector representations.",
    "Retrieval systems are used to find relevant documents for a query.",
    "RAG combines retrieval with text generation models.",
    "Precision measures how many retrieved results are relevant.",
    "Recall measures how many relevant results were retrieved.",
    "Clustering groups similar data points together.",
    "Classification assigns categories to input data.",
    "Regression predicts continuous values.",
    "Transformers are widely used in natural language processing.",
    "Tokenization splits text into smaller units.",
    "Vector search finds nearest neighbors in embedding space.",
    "Indexing improves search efficiency.",
    "Cosine similarity measures similarity between vectors.",
    "Evaluation metrics are essential for machine learning systems."
]

df = pd.DataFrame({"text": documents})
df.head()
print("Количество документов:", len(documents))
df.sample(5)

Количество документов: 20


,text
0,"Python is widely used in data science, machine..."
17,Indexing improves search efficiency.
15,Tokenization splits text into smaller units.
1,Pandas is a Python library used for data manip...
8,RAG combines retrieval with text generation mo...


## 2.3.3 Чанкинг документов

Документы разбиваются на чанки фиксированного размера (chunk_size=5 слов).

Такой размер выбран, чтобы:
- сохранить смысл фрагментов
- увеличить количество объектов для retrieval

Overlap не используется для простоты.

In [3]:
def chunk_text(text, chunk_size=5):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

chunks = []
for doc_id, text in enumerate(documents):
    for chunk in chunk_text(text):
        chunks.append((doc_id, chunk))

print("Пример документа:")
print(documents[0])

print("\nЧанки этого документа:")
print(chunk_text(documents[0]))

print("\nВсего чанков:", len(chunks))

chunks_df = pd.DataFrame(chunks, columns=["doc_id", "text"])
chunks_df.sample(5)

Пример документа:
Python is widely used in data science, machine learning, and automation tasks.

Чанки этого документа:
['Python is widely used in', 'data science, machine learning, and', 'automation tasks.']

Всего чанков: 43


,doc_id,text
23,9,Precision measures how many retrieved
34,15,Tokenization splits text into smaller
31,13,Regression predicts continuous values.
38,17,Indexing improves search efficiency.
9,3,by Facebook for efficient similarity


## 2.3.4. Эмбеддинги и индекс `FAISS`

In [4]:
texts = chunks_df["text"].tolist()

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts).toarray().astype("float32")

index = faiss.IndexFlatL2(X.shape[1])
index.add(X)

print("Размер индекса:", index.ntotal)

Размер индекса: 43


## 2.3.5 Контрольные запросы и оценка retrieval

In [5]:
def search(query, k=3):
    q_vec = vectorizer.transform([query]).toarray().astype("float32")
    D, I = index.search(q_vec, k)
    
    results = chunks_df.iloc[I[0]]
    return results.reset_index(drop=True)

search("What is FAISS?")
search("What is recall?")
search("What is machine learning?")

,doc_id,text
0,19,machine learning systems.
1,5,Deep learning is a subset
2,0,"data science, machine learning, and"


In [6]:
queries = [
    ("What is FAISS?", "faiss"),
    ("What is recall?", "recall"),
    ("What is precision?", "precision"),
    ("What are embeddings?", "embeddings"),
    ("What is machine learning?", "machine learning"),
    ("What is pandas?", "pandas"),
    ("What is numpy?", "numpy"),
    ("What is RAG?", "rag"),
    ("What is clustering?", "clustering"),
    ("What is regression?", "regression"),
]

results = []
k = 3

for q, expected in queries:
    res = search(q, k=k)
    retrieved = res["text"].tolist()
    
    hit = any(expected in r.lower() for r in retrieved)
    
    rank = next(
        (i+1 for i, r in enumerate(retrieved) if expected in r.lower()),
        None
    )
    
    results.append({
        "query": q,
        "expected_source": expected,
        "retrieved_sources": retrieved,
        "hit_at_k": hit,
        "rank_of_first_relevant": rank
    })

eval_df = pd.DataFrame(results)

hit_at_k = eval_df["hit_at_k"].mean()
recall_at_k = hit_at_k

print("Hit@k:", hit_at_k)
print("Recall@k:", recall_at_k)

eval_df.to_csv(
    "artifacts/retrieval_eval.csv",
    index=False
)

eval_df

Hit@k: 1.0
Recall@k: 1.0


,query,expected_source,retrieved_sources,hit_at_k,rank_of_first_relevant
0,What is FAISS?,faiss,"[FAISS is a library developed, Deep learning i...",True,1
1,What is recall?,recall,"[Recall measures how many relevant, Deep learn...",True,1
2,What is precision?,precision,"[Precision measures how many retrieved, Deep l...",True,1
3,What are embeddings?,embeddings,"[Embeddings convert text into dense, results a...",True,1
4,What is machine learning?,machine learning,"[machine learning systems., Deep learning is a...",True,1
5,What is pandas?,pandas,"[Pandas is a Python library, Deep learning is ...",True,1
6,What is numpy?,numpy,"[NumPy provides support for arrays, Deep learn...",True,1
7,What is RAG?,rag,"[RAG combines retrieval with text, Deep learni...",True,1
8,What is clustering?,clustering,"[Clustering groups similar data points, Deep l...",True,1
9,What is regression?,regression,"[Regression predicts continuous values., Deep ...",True,1


## 2.3.6 Эксперимент с параметрами retrieval

In [7]:
def evaluate_k(k):
    hits = []
    for q, expected in queries:
        res = search(q, k=k)
        retrieved = res["text"].tolist()
        hit = any(expected in r.lower() for r in retrieved)
        hits.append(hit)
    return np.mean(hits)

print("hit@1:", evaluate_k(1))
print("hit@3:", evaluate_k(3))
print("hit@5:", evaluate_k(5))

hit@1: 1.0
hit@3: 1.0
hit@5: 1.0


## 2.3.7. Обновление базы знаний и переиндексация

In [8]:
before_results = {
    q: search(q)["text"].tolist() for q, _ in queries
}

# добавляем новые документы
documents.extend([
    "FAISS supports GPU acceleration for faster search.",
    "Recall can be improved by increasing top_k.",
    "Precision decreases when too many irrelevant documents are retrieved."
])

In [9]:

new_docs = ["FAISS supports GPU acceleration.", "Pandas supports CSV files."]
documents.extend(new_docs)

# Rebuild
texts = []
for doc_id, text in enumerate(documents):
    for chunk in chunk_text(text):
        texts.append((doc_id, chunk))

chunks_df = pd.DataFrame(texts, columns=["doc_id", "text"])

X = vectorizer.fit_transform(chunks_df["text"]).toarray().astype("float32")
index = faiss.IndexFlatL2(X.shape[1])
index.add(X)

search("FAISS GPU")


,doc_id,text
0,23,FAISS supports GPU acceleration.
1,20,FAISS supports GPU acceleration for
2,3,FAISS is a library developed


In [10]:
# сравнение до/после
after_results = {
    q: search(q)["text"].tolist() for q, _ in queries
}

compare = []

for q in before_results:
    compare.append({
        "query": q,
        "before_retrieved_sources": before_results[q],
        "after_retrieved_sources": after_results[q],
        "changed": before_results[q] != after_results[q]
    })

compare_df = pd.DataFrame(compare)

compare_df.to_csv(
    "artifacts/retrieval_before_after_update.csv",
    index=False
)

compare_df

,query,before_retrieved_sources,after_retrieved_sources,changed
0,What is FAISS?,"[FAISS is a library developed, Deep learning i...","[FAISS is a library developed, FAISS supports ...",True
1,What is recall?,"[Recall measures how many relevant, Deep learn...","[Recall measures how many relevant, Recall can...",True
2,What is precision?,"[Precision measures how many retrieved, Deep l...","[Precision measures how many retrieved, Precis...",True
3,What are embeddings?,"[Embeddings convert text into dense, results a...","[Embeddings convert text into dense, results a...",True
4,What is machine learning?,"[machine learning systems., Deep learning is a...","[machine learning systems., Deep learning is a...",False
5,What is pandas?,"[Pandas is a Python library, Deep learning is ...","[Pandas is a Python library, Pandas supports C...",True
6,What is numpy?,"[NumPy provides support for arrays, Deep learn...","[NumPy provides support for arrays, Pandas is ...",True
7,What is RAG?,"[RAG combines retrieval with text, Deep learni...","[RAG combines retrieval with text, Pandas is a...",True
8,What is clustering?,"[Clustering groups similar data points, Deep l...","[Clustering groups similar data points, Pandas...",True
9,What is regression?,"[Regression predicts continuous values., Deep ...","[Regression predicts continuous values., Panda...",True


## 2.3.8 Mini-RAG

In [11]:
def rag(query):
    res = search(query, k=3)
    context = " ".join(res["text"].tolist())
    
    answer = context.split(".")[0]
    
    return answer, res["text"].tolist()

In [12]:
rag_examples = []

for q, _ in queries[:5]:
    ans, src = rag(q)
    rag_examples.append({
        "question": q,
        "answer": ans,
        "retrieved_sources": src
    })

rag_df = pd.DataFrame(rag_examples)

rag_df.to_csv(
    "artifacts/rag_examples.csv",
    index=False
)

rag_df

,question,answer,retrieved_sources
0,What is FAISS?,FAISS is a library developed FAISS supports GP...,"[FAISS is a library developed, FAISS supports ..."
1,What is recall?,Recall measures how many relevant Recall can b...,"[Recall measures how many relevant, Recall can..."
2,What is precision?,Precision measures how many retrieved Precisio...,"[Precision measures how many retrieved, Precis..."
3,What are embeddings?,Embeddings convert text into dense results are...,"[Embeddings convert text into dense, results a..."
4,What is machine learning?,machine learning systems,"[machine learning systems., Deep learning is a..."


## 2.3.9. Краткий анализ ошибок

Примеры работы mini-RAG показывают, что на простых definitional запросах (например, "What is FAISS?", "What is recall?") система возвращает корректные ответы, так как релевантный чанк попадает в top-3.

Однако наблюдаются следующие проблемы:
1. Частично нерелевантный retrieval  
Для запроса "What is recall?" в выдаче присутствует:
- "Pandas is a Python library"
- ошибка retrieval: в top-k попадают нерелевантные документы.
2. Потеря контекста  
Для запроса "What is machine learning?" ответ:
- "machine learning systems."  
- ошибка чанкинга: слишком короткие фрагменты не дают полного ответа.

3. Проблемы генерации  
Ответы формируются как склейка чанков:
- "FAISS is a library developed FAISS supports GPU acceleration"  
- ошибка генерации: нет нормальной агрегации текста.

Вывод: основные ошибки связаны с качеством retrieval (TF-IDF без семантики), маленьким размером чанков и простой схемой генерации ответа.